# [JAX·TPU 선택 심화] 01 · 사전학습 모델을 직접 읽고 한 장 추론하기
**목표:** `safetensors`로 DeiT 가중치를 불러오고 Pillow·NumPy로 이미지를 준비한 뒤 JAX로 원본 1,000개 클래스의 Top-5를 계산합니다.

Codespaces CPU에서 실제 추론합니다. GPU·TPU 실습 완료를 뜻하지 않습니다. 이 노트북은 모델 계산도 셀 안에 모두 보여 줍니다. 함수 정의를 읽는 데 오래 걸리면 제목과 라이브러리 호출부터 확인하고 각 셀을 순서대로 실행하세요.

기본 HF·PyTorch 과정은 `hf_colab_gpu/notebooks`에 있습니다. 이 심화 과정은 프로젝트 최상위에서 `bash scripts/setup.sh --with-jax`로 준비합니다.

## 라이브러리를 직접 불러오기
가상환경은 라이브러리 버전을 구분하는 공간입니다. 아래 `import`가 이 노트북에서 실제로 사용하는 라이브러리입니다.

| 가져오는 이름 | 설치할 패키지 | 하는 일 |
|---|---|---|
| `numpy` | `numpy` | 이미지 배열, 라벨, `.npz` 파일 |
| `PIL.Image` | `Pillow` | 이미지 읽기와 크기 조절 |
| `matplotlib.pyplot` | `matplotlib` | 이미지와 그래프 표시 |
| `IPython` | `ipykernel`과 함께 설치 | 노트북 안에 그래프 표시 |

`pathlib`, `os`, `sys`, `json`, `hashlib`, `importlib.metadata`는 Python 표준 라이브러리이므로 따로 설치하지 않습니다. 필요한 추가 라이브러리는 사용하는 셀에서 직접 불러옵니다.

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
from importlib.metadata import version

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/vision-ai")]
ROOT = next((path for path in candidates if (path / ".vision-lab-root").is_file()), None)
if ROOT is None:
    raise RuntimeError(".vision-lab-root가 있는 수업 폴더에서 열거나 Colab에 실습 파일을 먼저 업로드하세요.")
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
print("Project:", ROOT)
print("Python:", sys.executable)

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
for package in ("numpy", "Pillow", "matplotlib", "ipykernel"):
    print(f"{package}: {version(package)}")

### 저장된 데이터 읽기와 무결성 확인
`np.load(..., allow_pickle=False)`로 이미지·라벨·ID를 읽습니다. SHA256과 분할별 ID를 검사해 다른 데이터가 섞이거나 손상된 경우 중단합니다. 이 함수의 본문도 아래에 모두 표시합니다.

In [ ]:
def digest(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

In [ ]:
def load_prepared(path):
    path = Path(path)
    manifest = json.loads((path / "manifest.json").read_text(encoding="utf-8"))
    classes = manifest["classes"]
    if len(classes) < 2 or len(set(classes)) != len(classes):
        raise ValueError("클래스 목록이 잘못됐습니다.")
    splits, all_ids = {}, set()
    for name in ("train", "validation", "test"):
        record = manifest["splits"][name]
        if record["file"] != f"{name}.npz":
            raise ValueError("데이터 파일 경로가 예상 형식과 다릅니다.")
        file = path / record["file"]
        if digest(file) != record["sha256"]:
            raise ValueError(f"{name} 데이터 체크섬 불일치")
        with np.load(file, allow_pickle=False) as a:
            images, labels, ids = a["images"], a["labels"], a["ids"].tolist()
        if images.dtype != np.uint8 or images.ndim != 4 or images.shape[-1] != 3:
            raise ValueError("images는 uint8 NHWC RGB여야 합니다.")
        if len(images) != len(labels) or len(ids) != len(labels) or len(labels) != record["count"]:
            raise ValueError("이미지·라벨·식별자 개수가 다릅니다.")
        if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or len(labels) == 0 or labels.min() < 0 or labels.max() >= len(classes):
            raise ValueError("라벨 범위가 잘못됐습니다.")
        if len(set(ids)) != len(ids) or all_ids.intersection(ids):
            raise ValueError("분할 간 이미지 ID 중복")
        all_ids.update(ids)
        splits[name] = {"images": images, "labels": labels, "ids": ids}
    identity = {k: manifest[k] for k in ("classes", "splits", "seed", "preprocess")}
    if hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest() != manifest["dataset_sha256"]:
        raise ValueError("데이터 manifest 지문 불일치")
    return splits, manifest

In [ ]:
DATA_PATH = ROOT / 'data/prepared'
if not (DATA_PATH / "manifest.json").is_file():
    raise RuntimeError("00_setup_and_data.ipynb에서 데이터를 먼저 준비하세요.")
splits, manifest = load_prepared(DATA_PATH)
classes = manifest["classes"]
print("Classes:", classes)
print("Split counts:", {name: len(split["labels"]) for name, split in splits.items()})
print("Dataset fingerprint:", manifest["dataset_sha256"])

## 1. 예제 이미지 선택
기본 예시는 테스트 분할의 `cup`입니다. 여기서는 원본 모델의 라벨을 관찰할 뿐 최적 epoch를 고르거나 새 분류기의 정확도를 계산하지 않습니다.

In [ ]:
SAMPLE_CLASS = "cup"
label = classes.index(SAMPLE_CLASS)
sample_index = int(np.flatnonzero(splits["test"]["labels"] == label)[0])
raw = splits["test"]["images"][sample_index]
print("Image ID:", splits["test"]["ids"][sample_index])
plt.figure(figsize=(3, 3))
plt.imshow(raw, interpolation="nearest")
plt.title(f"Workbench label: {SAMPLE_CLASS}")
plt.axis("off")
plt.show()

### JAX와 실행 장치 확인
`jax`는 자동 미분과 컴파일을, `jax.numpy`는 배열 연산을 제공합니다. 설치 패키지는 CPU에서 `jax`, GPU에서 `jax[cuda12]`, TPU에서 `jax[tpu]`입니다. 같은 `import`를 사용하지만 실행 환경에 맞는 패키지가 필요합니다.

GPU·TPU를 요청했는데 장치가 없으면 오류로 중단합니다. CPU로 몰래 바꾸지 않습니다. 강사용 CPU 검증 때만 `VISION_DEVICE=cpu`를 명시할 수 있으며 출력에도 CPU로 기록됩니다.

In [ ]:
import jax
import jax.numpy as jnp

REQUESTED_DEVICE = os.environ.get("VISION_DEVICE", 'cpu')
if REQUESTED_DEVICE not in {"cpu", "gpu", "tpu"}:
    raise ValueError("VISION_DEVICE는 cpu, gpu, tpu 중 하나여야 합니다.")
try:
    devices = jax.devices(REQUESTED_DEVICE)
except RuntimeError as error:
    raise RuntimeError(f"{REQUESTED_DEVICE.upper()}를 찾지 못했습니다. 해당 Colab 런타임에서 실행하세요.") from error
if not devices or devices[0].platform != REQUESTED_DEVICE:
    raise RuntimeError("요청한 장치가 없습니다. Codespaces 자체에는 Colab GPU·TPU가 연결되지 않습니다.")
device = devices[0]
device_info = {"requested": REQUESTED_DEVICE, "platform": device.platform,
               "device_kind": device.device_kind, "available_count": len(devices),
               "used_count": 1, "jax_version": jax.__version__}
print("Device:", device_info)

### 모델 계산 1 · 행렬 곱과 정규화
`jnp.matmul`이 가중치와 입력을 곱합니다. LayerNorm은 마지막 축의 평균과 분산으로 값을 정규화합니다.

In [ ]:
def dense(params, prefix, x):
    return jnp.matmul(x, params[prefix + ".weight"].T,
                      precision=jax.lax.Precision.HIGHEST) + params[prefix + ".bias"]

def layer_norm(params, prefix, x, epsilon):
    centered = x - jnp.mean(x, axis=-1, keepdims=True)
    variance = jnp.mean(centered * centered, axis=-1, keepdims=True)
    return centered * jax.lax.rsqrt(variance + epsilon) * params[prefix + ".weight"] + params[prefix + ".bias"]

### 모델 계산 2 · Attention과 Transformer 블록
Query·Key의 내적으로 토큰 사이의 점수를 계산하고 `jax.nn.softmax`로 가중치를 만듭니다. 잔차 연결과 GELU까지 아래 함수에서 확인할 수 있습니다.

In [ ]:
def transformer_block(params, index, x, config):
    prefix = f"vit.encoder.layer.{index}"
    normalized = layer_norm(params, prefix + ".layernorm_before", x, config["layer_norm_eps"])
    batch, tokens, hidden = normalized.shape
    heads = config["num_attention_heads"]
    head_dim = hidden // heads

    def project(name):
        value = dense(params, prefix + ".attention.attention." + name, normalized)
        return value.reshape(batch, tokens, heads, head_dim).transpose(0, 2, 1, 3)

    query, key, value = (project(name) for name in ("query", "key", "value"))
    scores = jnp.matmul(query, key.swapaxes(-1, -2), precision=jax.lax.Precision.HIGHEST)
    probabilities = jax.nn.softmax(scores / jnp.sqrt(jnp.float32(head_dim)), axis=-1)
    context = jnp.matmul(probabilities, value, precision=jax.lax.Precision.HIGHEST)
    context = context.transpose(0, 2, 1, 3).reshape(batch, tokens, hidden)
    x = x + dense(params, prefix + ".attention.output.dense", context)
    normalized = layer_norm(params, prefix + ".layernorm_after", x, config["layer_norm_eps"])
    intermediate = jax.nn.gelu(dense(params, prefix + ".intermediate.dense", normalized), approximate=False)
    return x + dense(params, prefix + ".output.dense", intermediate)

### 모델 계산 3 · 이미지를 패치 토큰으로 바꾸기
224×224 이미지를 16×16 패치로 바꾼 뒤 첫 11개 블록을 통과시킵니다. 이 부분은 추가 학습에서 고정하므로 출력을 캐시할 수 있습니다.

In [ ]:
def prefix_tokens(params, images, config):
    """Patch embedding plus blocks 0..10; this output is safe to cache."""
    if images.ndim != 4 or images.shape[1:] != (224, 224, 3):
        raise ValueError("Expected preprocessed NHWC images [N, 224, 224, 3]")
    x = jax.lax.conv_general_dilated(
        images.astype(jnp.float32),
        params["vit.embeddings.patch_embeddings.projection.weight"].transpose(2, 3, 1, 0),
        window_strides=(config["patch_size"], config["patch_size"]),
        padding="VALID", dimension_numbers=("NHWC", "HWIO", "NHWC"),
        precision=jax.lax.Precision.HIGHEST,
    ) + params["vit.embeddings.patch_embeddings.projection.bias"]
    x = x.reshape(x.shape[0], -1, config["hidden_size"])
    cls = jnp.broadcast_to(params["vit.embeddings.cls_token"], (x.shape[0], 1, config["hidden_size"]))
    x = jnp.concatenate([cls, x], axis=1) + params["vit.embeddings.position_embeddings"]
    for index in range(config["num_hidden_layers"] - 1):
        x = transformer_block(params, index, x, config)
    return x

### 모델 계산 4 · 마지막 블록과 분류기
마지막 블록의 CLS 토큰으로 분류합니다. `original_logits`는 앞서 정의한 함수들을 순서대로 호출해 원본 1,000개 클래스 점수를 계산합니다.

In [ ]:
def tail_features(params, tokens, config):
    x = transformer_block(params, config["num_hidden_layers"] - 1, tokens, config)
    return layer_norm(params, "vit.layernorm", x, config["layer_norm_eps"])[:, 0]

def tail_logits(params, tokens, config):
    return dense(params, "classifier", tail_features(params, tokens, config))

def original_logits(params, images, config):
    return tail_logits(params, prefix_tokens(params, images, config), config)

### 학습할 가중치와 파일 지문
아래 함수는 마지막 블록·LayerNorm·분류기의 이름을 고르고 가중치가 바뀌었는지 확인합니다. `hashlib`의 SHA256은 파일과 배열의 지문을 만드는 표준 라이브러리 함수입니다.

In [ ]:
def is_tail_parameter(name, config):
    return (name.startswith(f"vit.encoder.layer.{config['num_hidden_layers'] - 1}.")
            or name.startswith("vit.layernorm.") or name.startswith("classifier."))

def extract_tail(params, config):
    return {key: value for key, value in params.items() if is_tail_parameter(key, config)}

def initialize_head(class_count, hidden_size, seed=42, device=None):
    if class_count < 2:
        raise ValueError("Fine-tuning requires at least two classes")
    weight = 0.02 * jax.random.normal(jax.random.PRNGKey(seed), (class_count, hidden_size))
    return {"classifier.weight": jax.device_put(weight, device),
            "classifier.bias": jax.device_put(jnp.zeros(class_count, jnp.float32), device)}

def parameter_digest(params):
    digest = hashlib.sha256()
    for key in sorted(params):
        value = np.ascontiguousarray(jax.device_get(params[key]))
        digest.update(key.encode("utf-8"))
        digest.update(str(value.shape).encode("ascii"))
        digest.update(value.dtype.str.encode("ascii"))
        digest.update(value.tobytes())
    return digest.hexdigest()

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

### 체크포인트 저장과 복원
`np.savez_compressed`와 `np.load`로 갱신한 마지막 블록·분류기를 저장하고 읽습니다. 복원할 때 원본 모델 revision과 각 배열 크기를 검사합니다.

In [ ]:
def save_checkpoint(path, tail, metadata):
    """Store only the adapted final block, norm and head; source stays unchanged."""
    arrays = {key: np.asarray(jax.device_get(value)) for key, value in tail.items()}
    arrays["__metadata__"] = np.array(json.dumps(metadata, ensure_ascii=False))
    np.savez_compressed(path, **arrays)

def load_checkpoint(path, source_params, config, device=None):
    with np.load(path, allow_pickle=False) as archive:
        metadata = json.loads(str(archive["__metadata__"]))
        if metadata.get("model_revision") != MODEL_REVISION:
            raise ValueError("Checkpoint revision does not match the pinned source")
        expected = set(extract_tail(source_params, config))
        actual = set(archive.files) - {"__metadata__"}
        if actual != expected:
            raise ValueError("Checkpoint tail parameter keys do not match the architecture")
        tail = {key: jax.device_put(archive[key], device) for key in actual}
    classes = metadata.get("classes", [])
    for key in expected:
        shape = ((len(classes), config["hidden_size"]) if key == "classifier.weight"
                 else (len(classes),) if key == "classifier.bias" else source_params[key].shape)
        if tail[key].shape != shape:
            raise ValueError(f"Checkpoint parameter shape mismatch: {key}")
    return {**source_params, **tail}, metadata

### safetensors로 사전학습 가중치 읽기
`safetensors.numpy.load_file`은 저장된 텐서를 NumPy 배열로 읽습니다. `jax.device_put`으로 이 배열들을 선택한 장치로 옮깁니다. 새 무작위 모델이 아니라 고정된 DeiT 가중치를 불러옵니다.

In [ ]:
from safetensors.numpy import load_file

MODEL_ID = "facebook/deit-tiny-patch16-224"
MODEL_REVISION = "b3428f18dcc7b543470d07f14b4a4157815d1880"
MODEL_SHA256 = "056550dbe6c439dddb35e1800a48aaef86cfdcf8e566ba73f0d1ce41bc7fa1b3"
ASSETS = ROOT / "assets/pretrained"
config = json.loads((ASSETS / "config.json").read_text(encoding="utf-8"))
if (config["hidden_act"] != "gelu" or config["num_hidden_layers"] != 12
        or config["hidden_size"] != 192 or config["image_size"] != 224
        or config["patch_size"] != 16 or config["num_attention_heads"] != 3):
    raise ValueError("고정된 DeiT tiny 모델 구조와 다릅니다.")
weights_path = ASSETS / "model.safetensors"
if file_sha256(weights_path) != MODEL_SHA256:
    raise ValueError("사전학습 파일 SHA256이 다릅니다.")
numpy_weights = load_file(str(weights_path))
if numpy_weights["classifier.weight"].shape != (1000, 192):
    raise ValueError("원본 1,000개 클래스 분류기가 필요합니다.")
params = {name: jax.device_put(np.asarray(value, np.float32), device)
          for name, value in numpy_weights.items()}
del numpy_weights
print("Model:", MODEL_ID, "revision:", MODEL_REVISION)
print("safetensors:", version("safetensors"), "tensors:", len(params))
for name in ["vit.embeddings.patch_embeddings.projection.weight",
             "vit.encoder.layer.11.attention.attention.query.weight", "classifier.weight"]:
    print(name, params[name].shape, params[name].dtype)

### Pillow와 NumPy로 입력 전처리
RGB 이미지를 224×224로 바꾸고 픽셀 값을 `(pixel / 255 - 0.5) / 0.5`로 변환합니다. 모델의 `preprocessor_config.json`과 같은 설정입니다. `preprocess`는 외부 패키지가 아닌 아래 셀에서 직접 정의하는 함수입니다.

In [ ]:
def preprocess(images):
    if not len(images):
        return np.empty((0, 224, 224, 3), dtype=np.float32)
    resized = []
    for pixels in images:
        image = Image.fromarray(np.asarray(pixels, dtype=np.uint8)).convert("RGB")
        image = image.resize((224, 224), Image.Resampling.BILINEAR)
        resized.append(np.asarray(image, dtype=np.float32))
    return (np.stack(resized) / np.float32(255) - np.float32(0.5)) / np.float32(0.5)

## 2. 전처리한 배열을 CPU에 올리기
출력 크기는 `(1, 224, 224, 3)`, 자료형은 `float32`입니다. `jax.device_put`은 계산에 사용할 장치로 배열을 보냅니다.

In [ ]:
pixels = jax.device_put(preprocess(raw[None, ...]), device)
print("Input:", pixels.shape, pixels.dtype)
print("Original classifier:", params["classifier.weight"].shape)

## 3. jax.jit으로 컴파일하고 추론하기
`jax.jit`은 위에서 정의한 모델 계산을 컴파일합니다. 첫 호출에는 컴파일 시간이 포함됩니다. softmax 값은 원본 라벨 체계 안의 상대적인 점수이며 실제 작업대 환경에서 보정한 신뢰도가 아닙니다.

In [ ]:
forward = jax.jit(lambda weights, images: original_logits(weights, images, config))
logits = forward(params, pixels)
probabilities = np.asarray(jax.device_get(jax.nn.softmax(logits[0])))
top = np.argsort(probabilities)[-5:][::-1]
top_labels = [config["id2label"][str(int(index))] for index in top]
assert logits.shape == (1, 1000) and np.isfinite(probabilities).all()
for name, score in zip(top_labels, probabilities[top]):
    print(f"{float(score):.3f}  {name}")

## 4. Top-5 표시하기

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
positions = np.arange(len(top))
ax.barh(positions, probabilities[top], color="#4285F4")
ax.set_yticks(positions, top_labels)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("Softmax probability")
ax.set_title("Original pretrained model · top 5 ImageNet labels · one test image")
for y, score in zip(positions, probabilities[top]):
    ax.text(min(float(score) + 0.01, 0.94), y, f"{float(score):.3f}", va="center")
fig.tight_layout()
plt.show()

## 확인 질문과 다음 단계
- `load_file`, `Image.resize`, `jax.jit`, `jax.nn.softmax`는 각각 어느 단계에서 사용했나요?
- 우리 라벨 `cup`과 원본 모델의 `coffee mug`는 어떻게 비교해야 할까요?
- 새 5개 클래스 분류기를 만들면 그 분류기는 이미 학습된 상태일까요?

다음에는 새 분류기를 먼저 학습해 기준 모델을 만들고 마지막 사전학습 블록도 업데이트합니다. 원본 1,000개 클래스의 결과를 새 5개 클래스 정확도와 같은 지표로 비교하지 않습니다.

모델 출처: [DeiT](https://huggingface.co/facebook/deit-tiny-patch16-224/tree/b3428f18dcc7b543470d07f14b4a4157815d1880).